# Part 2: Multi-Source RAG with Routing

## 1. Setup and Configuration

In [ ]:
import subprocess
import os
import json
import re
import glob
import pandas as pd
from dotenv import load_dotenv
from litellm import completion

load_dotenv()

MODEL = 'groq/llama-3.3-70b-versatile'
CSV_PATH = os.path.join('data', 'structured', 'daily_sales.csv')
TEXT_DIR = os.path.join('data', 'unstructured')

assert os.path.isfile(CSV_PATH), f'CSV not found at {CSV_PATH}'
assert os.path.isdir(TEXT_DIR), f'Text directory not found at {TEXT_DIR}'
assert os.getenv('GROQ_API_KEY'), 'GROQ_API_KEY not set'
print(f'✓ CSV: {CSV_PATH}')
print(f'✓ Text dir: {TEXT_DIR}')
print(f'✓ Model: {MODEL}')

✓ CSV: data/structured/daily_sales.csv
✓ Text dir: data/unstructured
✓ Model: groq/llama-3.3-70b-versatile


## 2. Data Exploration


### 2.1 Structured Data (CSV)

In [2]:
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nData types:')
print(df.dtypes)
print(f'\nFirst 5 rows:')
df.head()

Shape: (1000, 8)

Columns: ['date', 'product_id', 'product_name', 'category', 'units_sold', 'unit_price', 'total_revenue', 'region']

Data types:
date                 str
product_id           str
product_name         str
category             str
units_sold         int64
unit_price       float64
total_revenue    float64
region               str
dtype: object

First 5 rows:


,date,product_id,product_name,category,units_sold,unit_price,total_revenue,region
0,2024-10-03,BOOK003,Mystery Novel Collection,Books,42,24.99,1049.58,Central
1,2024-10-03,ELEC002,USB-C Fast Charger,Electronics,57,24.99,1424.43,Central
2,2024-10-03,BOOK001,Python Programming Guide,Books,39,39.65,1546.35,West
3,2024-10-03,ELEC003,Portable Power Bank 20000mAh,Electronics,22,49.99,1099.78,North
4,2024-10-03,FOOD001,Organic Coffee Beans 1kg,Food & Grocery,11,24.99,274.89,North


In [3]:
print('=== Key Statistics ===')
print(f'\nDate range: {df["date"].min()} to {df["date"].max()}')
print(f'\nCategories ({df["category"].nunique()}):')
print(df['category'].value_counts())
print(f'\nRegions ({df["region"].nunique()}):')
print(df['region'].value_counts())
print(f'\nProducts: {df["product_id"].nunique()} unique')
print(f'Total revenue: ${df["total_revenue"].sum():,.2f}')

=== Key Statistics ===

Date range: 2024-10-03 to 2024-12-31

Categories (10):
category
Electronics               140
Clothing                  130
Office Supplies           107
Pet Supplies              100
Sports & Outdoors          98
Books                      96
Beauty & Personal Care     85
Home & Kitchen             83
Food & Grocery             82
Toys & Games               79
Name: count, dtype: int64

Regions (5):
region
Central    234
East       202
West       199
North      190
South      175
Name: count, dtype: int64

Products: 34 unique
Total revenue: $1,738,661.87


In [ ]:
print('=== Product ID → Name Mapping (sample) ===')
product_map = df[['product_id', 'product_name', 'category']].drop_duplicates().sort_values('product_id')
print(product_map.to_string(index=False))

=== Product ID → Name Mapping (sample) ===
product_id                  product_name               category
   BEAU001               Vitamin C Serum Beauty & Personal Care
   BEAU002       Hair Dryer Professional Beauty & Personal Care
   BEAU003           Electric Toothbrush Beauty & Personal Care
   BOOK001      Python Programming Guide                  Books
   BOOK002            The Art of Cooking                  Books
   BOOK003      Mystery Novel Collection                  Books
   CLTH001             Running Shoes Men               Clothing
   CLTH002           Winter Jacket Women               Clothing
   CLTH003       Cotton T-Shirt Pack (3)               Clothing
   CLTH004           Denim Jeans Classic               Clothing
   ELEC001 Wireless Bluetooth Headphones            Electronics
   ELEC002            USB-C Fast Charger            Electronics
   ELEC003  Portable Power Bank 20000mAh            Electronics
   ELEC004               Smart Watch Pro            Electroni

### 2.2 Unstructured Data (Text Files)

In [ ]:
text_files = sorted(glob.glob(os.path.join(TEXT_DIR, '*.txt')))
print(f'Found {len(text_files)} text files:\n')
for f in text_files:
    basename = os.path.basename(f)
    with open(f) as fh:
        content = fh.read()
    print(f'  {basename}: {len(content)} chars, {len(content.split())} words')

print(f'\n=== Preview: {os.path.basename(text_files[0])} (first 800 chars) ===')
with open(text_files[0]) as fh:
    print(fh.read()[:800])

Found 10 text files:

  BEAU001_product_page.txt: 2833 chars, 454 words
  BOOK001_product_page.txt: 2740 chars, 410 words
  CLTH001_product_page.txt: 2543 chars, 391 words
  ELEC001_product_page.txt: 2325 chars, 349 words
  FOOD001_product_page.txt: 2732 chars, 428 words
  HOME003_product_page.txt: 2437 chars, 392 words
  OFFC001_product_page.txt: 2674 chars, 407 words
  PETS001_product_page.txt: 2694 chars, 438 words
  SPRT001_product_page.txt: 2387 chars, 387 words
  TOYS001_product_page.txt: 2566 chars, 405 words

=== Preview: BEAU001_product_page.txt (first 800 chars) ===
VITAMIN C SERUM - PRODUCT PAGE

Product: Vitamin C Serum
Brand: GlowLab Skincare
Price: $28.99
SKU: BEAU001
Category: Beauty & Personal Care

PRODUCT DESCRIPTION:
Reveal brighter, more youthful skin with GlowLab's Vitamin C Serum. This powerful
antioxidant formula combines 20% Vitamin C (L-Ascorbic Acid) with Vitamin E and
Hyaluronic Acid to reduce dark spots, fine lines, and sun damage while hydrating
and protect

In [ ]:
text_data = {}
for filepath in text_files:
    pid = os.path.basename(filepath).replace('_product_page.txt', '')
    with open(filepath) as fh:
        text_data[pid] = {
            'filepath': filepath,
            'filename': os.path.basename(filepath),
            'content': fh.read()
        }

print(f'Loaded {len(text_data)} product pages: {list(text_data.keys())}')

Loaded 10 product pages: ['BEAU001', 'BOOK001', 'CLTH001', 'ELEC001', 'FOOD001', 'HOME003', 'OFFC001', 'PETS001', 'SPRT001', 'TOYS001']


### 2.3 Data Exploration Summary

**CSV:** 1000 rows, 8 columns (date, product_id, product_name, category, units_sold, unit_price, total_revenue, region). Covers Oct-Dec 2024 across 5 regions and 10 categories.

**Text:** 10 product pages with descriptions, specs, and customer reviews. Product IDs map to the CSV data.

This tells us:
- Numerical questions (revenue, top seller, volumes) → **CSV with pandas**
- Feature/review questions → **Text search**
- Combined questions (highly rated AND best selling) → **Both sources**

## 3. Query Router

In [7]:
def route_query(question: str) -> dict:
    """Classify an e-commerce question and determine data source routing."""
    prompt = """You are a query router for an e-commerce analytics system with two data sources:

1. CSV data: daily_sales.csv with columns [date, product_id, product_name, category, units_sold, unit_price, total_revenue, region]
   - 1000 rows of sales data from Oct-Dec 2024
   - Categories: Electronics, Home & Kitchen, Sports, Beauty, Clothing, Books, Toys, Office, Pets, Food
   - Regions: North, South, East, West, Central

2. Text files: Product pages with descriptions, specifications, and customer reviews
   - 10 product files (ELEC001, HOME003, SPRT001, BEAU001, CLTH001, BOOK001, TOYS001, OFFC001, PETS001, FOOD001)

Classify this question and output ONLY a JSON object with:
- "route": one of ["csv", "text", "both"]
  - "csv" for numerical/analytical questions (revenue, totals, rankings by sales, volumes, comparisons by region/date)
  - "text" for product features, descriptions, customer reviews, opinions, specifications
  - "both" for questions needing both sales performance data AND product details/reviews
- "csv_query_description": plain English description of what to compute from the CSV (null if route is "text")
- "text_keywords": list of keywords to search in product descriptions (null if route is "csv")
- "relevant_product_ids": list of product IDs if specific products are mentioned (empty list if none)
- "reasoning": one sentence explaining the routing decision

Question: {question}

JSON only, no markdown:"""

    response = completion(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt.format(question=question)}],
        temperature=0
    )
    
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {'route': 'both', 'csv_query_description': question, 'text_keywords': question.split(), 'relevant_product_ids': [], 'reasoning': 'fallback'}

### 3.1 Test the Router

In [8]:
test_questions_p2 = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?"
]

expected_routes = ['csv', 'csv', 'text', 'text', 'both', 'both']

for i, (q, expected) in enumerate(zip(test_questions_p2, expected_routes), 1):
    routing = route_query(q)
    actual = routing.get('route', '?')
    match = '✓' if actual == expected else '✗'
    print(f'{match} Q{i}: route={actual} (expected={expected})')
    print(f'    {q[:70]}')
    print(f'    Reasoning: {routing.get("reasoning", "N/A")}')
    print()

✓ Q1: route=csv (expected=csv)
    What was the total revenue for Electronics category in December 2024?
    Reasoning: This question requires numerical data from the CSV file to calculate total revenue for a specific category and time period.

✓ Q2: route=csv (expected=csv)
    Which region had the highest sales volume?
    Reasoning: This question requires numerical analysis of sales data, which is available in the CSV file.

✓ Q3: route=text (expected=text)
    What are the key features of the Wireless Bluetooth Headphones?
    Reasoning: The question asks for product features, which are typically found in product descriptions, so it is routed to the text data source.

✓ Q4: route=text (expected=text)
    What do customers say about the Air Fryer's ease of cleaning?
    Reasoning: This question is routed to the text data source because it asks about customer opinions and reviews of a specific product feature.

✓ Q5: route=both (expected=both)
    Which product has the best customer 

## 4. CSV Retrieval (Structured Data)

In [ ]:
def retrieve_csv(question: str, csv_query_description: str = None) -> str:
    """Generate and execute pandas code to answer a question from the CSV."""
    query_desc = csv_query_description or question
    
    code_prompt = f"""You have a pandas DataFrame `df` loaded from a sales CSV.

DataFrame info:
- Columns: {df.columns.tolist()}
- dtypes: date(str, format YYYY-MM-DD), product_id(str), product_name(str), category(str), units_sold(int), unit_price(float), total_revenue(float), region(str)
- {len(df)} rows, dates from {df['date'].min()} to {df['date'].max()}
- Categories: {df['category'].unique().tolist()}
- Regions: {df['region'].unique().tolist()}
- Products: {df[['product_id','product_name']].drop_duplicates().values.tolist()[:15]}...

Task: {query_desc}

Write Python code using `df` (already loaded) and `pd` (already imported).
Store the final answer in a variable called `result`.
Make `result` a descriptive string that includes the actual numbers/data.

Output ONLY executable Python code. No markdown, no explanation, no imports."""

    response = completion(
        model=MODEL,
        messages=[{'role': 'user', 'content': code_prompt}],
        temperature=0
    )
    
    code = response.choices[0].message.content.strip()
    code = re.sub(r'^```python\s*', '', code)
    code = re.sub(r'\s*```$', '', code)
    
    print(f'  [CSV] Generated code:\n{code}\n')
    
    local_vars = {'df': df.copy(), 'pd': pd}
    try:
        exec(code, {'__builtins__': __builtins__}, local_vars)
        result = local_vars.get('result', 'No result variable found')
        print(f'  [CSV] Result: {result}')
        return str(result)
    except Exception as e:
        error_msg = f'Code execution error: {e}'
        print(f'  [CSV] {error_msg}')
        retry_prompt = f'The previous code failed with: {e}\n\nFix the code. Original task: {query_desc}\n\nDataFrame columns: {df.columns.tolist()}\nOutput ONLY executable Python code.'
        response2 = completion(
            model=MODEL,
            messages=[{'role': 'user', 'content': retry_prompt}],
            temperature=0
        )
        code2 = response2.choices[0].message.content.strip()
        code2 = re.sub(r'^```python\s*', '', code2)
        code2 = re.sub(r'\s*```$', '', code2)
        try:
            exec(code2, {'__builtins__': __builtins__}, local_vars)
            result = local_vars.get('result', 'No result after retry')
            print(f'  [CSV] Retry result: {result}')
            return str(result)
        except Exception as e2:
            return f'Failed to query CSV: {e2}'

### 4.1 Test CSV Retrieval

In [ ]:
print('Test: Electronics revenue in December 2024')
csv_result = retrieve_csv(test_questions_p2[0], 'Calculate total revenue for Electronics category in December 2024')
print(f'\nFinal result: {csv_result}')
print()

dec_elec = df[(df['category'] == 'Electronics') & (df['date'] >= '2024-12-01') & (df['date'] <= '2024-12-31')]
print(f'Manual verification: ${dec_elec["total_revenue"].sum():,.2f}')

Test: Electronics revenue in December 2024
  [CSV] Generated code:
december_2024 = df[(df['date'].str.startswith('2024-12')) & (df['category'] == 'Electronics')]
result = f"Total revenue for Electronics category in December 2024: ${december_2024['total_revenue'].sum():.2f}"

  [CSV] Result: Total revenue for Electronics category in December 2024: $142864.31

Final result: Total revenue for Electronics category in December 2024: $142864.31

Manual verification: $142,864.31


In [ ]:
print('Test: Region with highest sales volume')
csv_result2 = retrieve_csv(test_questions_p2[1], 'Find which region had the highest total sales volume (sum of units_sold)')
print(f'\nFinal result: {csv_result2}')
print()

print('Manual verification:')
print(df.groupby('region')['units_sold'].sum().sort_values(ascending=False))

Test: Region with highest sales volume
  [CSV] Generated code:
result = df.groupby('region')['units_sold'].sum().idxmax() + ' had the highest total sales volume with ' + str(df.groupby('region')['units_sold'].sum().max()) + ' units sold'

  [CSV] Result: Central had the highest total sales volume with 6779 units sold

Final result: Central had the highest total sales volume with 6779 units sold

Manual verification:
region
Central    6779
East       5678
North      5610
West       5423
South      5314
Name: units_sold, dtype: int64


## 5. Text Retrieval (Unstructured Data)

In [ ]:
def retrieve_text(question: str, keywords: list = None, product_ids: list = None, top_k: int = 3) -> str:
    """Search product text files using keyword matching and return relevant content."""
    if not keywords:

        keywords = [w.lower() for w in question.split() if len(w) > 3]
    else:
        keywords = [k.lower() for k in keywords]
    
    scored_results = []
    
    for pid, data in text_data.items():
        content_lower = data['content'].lower()
        
        score = 0
        matched_keywords = []
        for kw in keywords:
            count = content_lower.count(kw.lower())
            if count > 0:
                score += count
                matched_keywords.append(kw)
        
        if product_ids and pid in product_ids:
            score += 100
        
        if score > 0:
            scored_results.append({
                'product_id': pid,
                'filename': data['filename'],
                'content': data['content'],
                'score': score,
                'matched_keywords': matched_keywords
            })
    
    scored_results.sort(key=lambda x: x['score'], reverse=True)
    
    top_results = scored_results[:top_k]
    
    if not top_results:
        return 'No matching product pages found for the given keywords.'

    context_parts = []
    for r in top_results:
        print(f'  [TEXT] {r["filename"]} (score: {r["score"]}, keywords: {r["matched_keywords"]})')
        context_parts.append(f'=== {r["filename"]} (product: {r["product_id"]}) ===\n{r["content"]}')
    
    return '\n\n'.join(context_parts)

### 5.1 Test Text Retrieval

In [ ]:
print('Test: Key features of Wireless Bluetooth Headphones')
text_result = retrieve_text(
    test_questions_p2[2], 
    keywords=['wireless', 'bluetooth', 'headphones', 'features']
)
print(f'\nRetrieved {len(text_result)} characters of context')
print(f'Preview: {text_result[:300]}...')

Test: Key features of Wireless Bluetooth Headphones
  [TEXT] ELEC001_product_page.txt (score: 14, keywords: ['wireless', 'bluetooth', 'headphones', 'features'])
  [TEXT] OFFC001_product_page.txt (score: 2, keywords: ['features'])
  [TEXT] CLTH001_product_page.txt (score: 1, keywords: ['features'])

Retrieved 7702 characters of context
Preview: === ELEC001_product_page.txt (product: ELEC001) ===
WIRELESS BLUETOOTH HEADPHONES - PRODUCT PAGE

Product: Wireless Bluetooth Headphones
Brand: SoundMax Pro
Price: $79.99
SKU: ELEC001
Category: Electronics

PRODUCT DES...


In [ ]:
print('Test: Air Fryer ease of cleaning')
text_result2 = retrieve_text(
    test_questions_p2[3], 
    keywords=['air fryer', 'cleaning', 'clean', 'easy', 'maintenance']
)
print(f'\nRetrieved {len(text_result2)} characters of context')
print(f'Preview: {text_result2[:300]}...')

Test: Air Fryer ease of cleaning
  [TEXT] HOME003_product_page.txt (score: 10, keywords: ['air fryer', 'clean', 'easy'])
  [TEXT] SPRT001_product_page.txt (score: 6, keywords: ['cleaning', 'clean', 'easy'])
  [TEXT] BEAU001_product_page.txt (score: 2, keywords: ['clean', 'easy'])

Retrieved 7817 characters of context
Preview: === HOME003_product_page.txt (product: HOME003) ===
AIR FRYER 5.5L - PRODUCT PAGE

Product: Air Fryer 5.5L
Brand: KitchenPro Elite
Price: $129.99
SKU: HOME003
Category: Home & Kitchen

PRODUCT DESCRIPTION:
Transform yo...


## 6. Answer Generation

In [15]:
def generate_answer_p2(question: str, csv_context: str = None, text_context: str = None) -> str:
    """Generate an answer using the LLM with context from CSV and/or text sources."""
    context_sections = []
    if csv_context:
        context_sections.append(f'## Sales Data (CSV Analysis):\n{csv_context}')
    if text_context:
        context_sections.append(f'## Product Information (Text Files):\n{text_context}')
    
    combined_context = '\n\n'.join(context_sections)
    
    prompt = f"""You are an e-commerce analytics assistant. Answer the question using ONLY the provided data.

RULES:
- Cite specific data points (numbers, product names, file names)
- If you have sales data, include actual revenue figures or unit counts
- If you have reviews, reference specific customer feedback
- Be concise but complete

{combined_context}

Question: {question}

Answer:"""
    
    response = completion(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.1,
        max_tokens=1500
    )
    return response.choices[0].message.content

## 7. Complete Pipeline

In [ ]:
def multi_source_qa(question: str, verbose: bool = True) -> str:
    """Full pipeline: route → retrieve from source(s) → generate answer."""
    
    if verbose:
        print(f'\n{"="*80}')
        print(f'QUESTION: {question}')
        print(f'{"="*80}')
    
    routing = route_query(question)
    route = routing.get('route', 'both')
    if verbose:
        print(f'\n[1] Route: {route}')
        print(f'    Reasoning: {routing.get("reasoning", "N/A")}')
    
    csv_context = None
    text_context = None

    if route in ('csv', 'both'):
        if verbose:
            print(f'\n[2a] Querying CSV...')
        csv_desc = routing.get('csv_query_description', question)
        csv_context = retrieve_csv(question, csv_desc)
    
    if route in ('text', 'both'):
        if verbose:
            print(f'\n[2b] Searching text files...')
        text_keywords = routing.get('text_keywords', None)
        product_ids = routing.get('relevant_product_ids', [])
        text_context = retrieve_text(question, text_keywords, product_ids)
    
    if verbose:
        print(f'\n[3] Generating answer...')
        if csv_context:
            print(f'    CSV context: {len(str(csv_context))} chars')
        if text_context:
            print(f'    Text context: {len(str(text_context))} chars')
    
    answer = generate_answer_p2(question, csv_context, text_context)
    
    if verbose:
        print(f'\n[4] ANSWER:')
        print(answer)
    
    return answer

## 8. Run All Test Questions

In [17]:
results_p2 = {}

for i, question in enumerate(test_questions_p2, 1):
    print(f'\n{"#"*80}')
    print(f'# Question {i} of {len(test_questions_p2)}')
    print(f'{"#"*80}')
    
    answer = multi_source_qa(question, verbose=True)
    results_p2[i] = {'question': question, 'answer': answer}
    print(f'\n--- Q{i} complete ---')


################################################################################
# Question 1 of 6
################################################################################

QUESTION: What was the total revenue for Electronics category in December 2024?

[1] Route: csv
    Reasoning: This question requires numerical data from the CSV file to calculate total revenue for a specific category and time period.

[2a] Querying CSV...
  [CSV] Generated code:
december_2024 = df[(df['date'].str.startswith('2024-12')) & (df['category'] == 'Electronics')]
result = f"Total revenue for Electronics category in December 2024: ${december_2024['total_revenue'].sum():.2f}"

  [CSV] Result: Total revenue for Electronics category in December 2024: $142864.31

[3] Generating answer...
    CSV context: 67 chars

[4] ANSWER:
The total revenue for the Electronics category in December 2024 was $142,864.31.

--- Q1 complete ---

############################################################################

## 9. Save Results

In [18]:
with open('part2_results.txt', 'w') as f:
    f.write('Part 2: Multi-Source RAG with Routing - Results\n')
    f.write('=' * 80 + '\n\n')
    
    for i, data in results_p2.items():
        f.write(f'Question {i}: {data["question"]}\n')
        f.write('-' * 60 + '\n')
        f.write(f'{data["answer"]}\n\n')
        f.write('=' * 80 + '\n\n')

print('✓ Results saved to part2_results.txt')
print(f'  Total questions answered: {len(results_p2)}')

✓ Results saved to part2_results.txt
  Total questions answered: 6


## 10. Quality Review

In [ ]:
expected = {1: 'csv', 2: 'csv', 3: 'text', 4: 'text', 5: 'both', 6: 'both'}

for i, data in results_p2.items():
    answer = data['answer']
    word_count = len(answer.split())
    has_numbers = bool(re.search(r'\$[\d,]+|\d+\.\d+|\d{3,}', answer))
    has_product_refs = bool(re.search(r'[A-Z]{4}\d{3}|product|headphone|fryer', answer, re.I))
    print(f'Q{i} (expected route: {expected[i]}): {word_count} words | Numbers: {"✓" if has_numbers else "~"} | Product refs: {"✓" if has_product_refs else "~"}')

print('\nReview complete. Check answers for accuracy against the data.')

Q1 (expected route: csv): 12 words | Numbers: ✓ | Product refs: ~
Q2 (expected route: csv): 12 words | Numbers: ✓ | Product refs: ~
Q3 (expected route: text): 66 words | Numbers: ✓ | Product refs: ✓
Q4 (expected route: text): 63 words | Numbers: ✓ | Product refs: ✓
Q5 (expected route: both): 35 words | Numbers: ✓ | Product refs: ✓
Q6 (expected route: both): 46 words | Numbers: ✓ | Product refs: ✓

Review complete. Check answers for accuracy against the data.
